# Introduction to Arabic Speech Technologies
## Chapter 3 notebook: From the waveform to model-ready features

Electronic supplementary material for *Introduction to Arabic Speech Technologies* by Hend S. Al-Khalifa.

This notebook implements the formal definitions that Chapter 3 states in words and arithmetic: the short-time Fourier transform and the spectrogram, the mel filterbank, the cepstral coefficients (MFCCs) and linear prediction. Every step is written out with `numpy` so that the exact computation can be inspected, and the intermediate sizes match the ones quoted in the chapter (16 kHz audio, 25 ms frames, 10 ms hop, a 512-point transform, 257 power bins, 40 mel filters, 13 cepstral coefficients, 39 with deltas).

**Contents**

1. Audio in: a synthetic Arabic-like vowel, or your own WAV file
2. Sampling, quantization and the Nyquist limit
3. Framing and windowing
4. The short-time Fourier transform and two spectrogram settings
5. The mel filterbank and log-mel features
6. The discrete cosine transform and MFCCs, with deltas
7. Linear prediction: autocorrelation, Levinson-Durbin, and formants
8. Telephone-band simulation (the effect shown in Figure 3.8)
9. Optional: a clip from Common Voice Arabic

**Running it.** The notebook needs only `numpy`, `scipy` and `matplotlib` (see `requirements.txt`). It runs top to bottom with no downloads and no audio files: where a recording is useful, the notebook synthesises one, and a cell is provided for reading your own WAV file instead. Optional cells that need extra packages or internet access are marked *Optional*.


## 1. Audio in

The cells below build a 0.6 s synthetic vowel at 16 kHz: a glottal-like pulse train at a fundamental frequency of 120 Hz, shaped by three resonances (formants). Synthetic audio keeps the notebook runnable offline and makes the formant values known in advance, which is useful when checking the linear-prediction section. To work with a real recording instead, run the cell marked *your own file*.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter, resample_poly, butter, sosfilt

SR = 16000            # sampling rate in Hz
FRAME_MS, HOP_MS = 25, 10
N_FFT, N_MELS, N_MFCC = 512, 40, 13
rng = np.random.default_rng(0)

def formant_filter(x, formants, bandwidths, sr=SR):
    """Cascade of two-pole resonators, one per formant."""
    y = x.copy()
    for f, b in zip(formants, bandwidths):
        r = np.exp(-np.pi * b / sr)
        theta = 2 * np.pi * f / sr
        a = [1.0, -2 * r * np.cos(theta), r ** 2]
        y = lfilter([1.0 - 2 * r * np.cos(theta) + r ** 2], a, y)
    return y

def synth_vowel(f0=120.0, dur=0.6, formants=(700, 1220, 2600), bandwidths=(80, 90, 120), sr=SR):
    n = int(dur * sr)
    t = np.arange(n) / sr
    pulses = np.zeros(n)
    pulses[:: int(sr / f0)] = 1.0                 # glottal pulse train
    x = formant_filter(pulses, formants, bandwidths, sr)
    x += 0.001 * rng.standard_normal(n)           # a little breath noise
    window = np.minimum(1.0, np.minimum(t / 0.05, (dur - t) / 0.05))  # fade in and out
    x *= np.clip(window, 0, 1)
    return (x / np.max(np.abs(x))).astype(np.float32)

def synth_token(sr=SR):
    # A stop release followed by a vowel, roughly the shape of تين (tin, 'figs').
    burst_n = int(0.03 * sr)
    noise = rng.standard_normal(burst_n)
    hp = butter(4, 2500 / (sr / 2), btype="high", output="sos")
    burst = sosfilt(hp, noise) * np.exp(-np.arange(burst_n) / (0.004 * sr))
    burst /= np.max(np.abs(burst))
    silence = np.zeros(int(0.04 * sr))
    vowel = synth_vowel(sr=sr)
    token = np.concatenate([silence, 0.6 * burst, vowel])
    return (token / np.max(np.abs(token))).astype(np.float32)

x = synth_token()
print(f"{len(x)} samples, {len(x)/SR:.2f} s at {SR} Hz")
print("The token is a high-frequency release burst followed by a voiced vowel,")
print("so it carries energy both inside and above the telephone band (section 8).")

10720 samples, 0.67 s at 16000 Hz
The token is a high-frequency release burst followed by a voiced vowel,
so it carries energy both inside and above the telephone band (section 8).


*Your own file.* Uncomment and point the path at a mono WAV recording, for example one of the minimal-pair recordings discussed in Chapter 2 (تين *tīn* and طين *ṭīn*). Anything not already at 16 kHz is resampled, and a stereo file is mixed down to one channel.

In [2]:
# from scipy.io import wavfile
# sr_in, raw = wavfile.read("my_recording.wav")
# raw = raw.astype(np.float64)
# if raw.ndim > 1: raw = raw.mean(axis=1)
# if sr_in != SR: raw = resample_poly(raw, SR, sr_in)
# x = (raw / np.max(np.abs(raw))).astype(np.float32)
# print(f"{len(x)} samples, {len(x)/SR:.2f} s at {SR} Hz")

## 2. Sampling, quantization and the Nyquist limit

Sampling replaces a continuous wave with measurements taken at a fixed rate; quantization rounds each measurement to one of a finite set of levels. The Nyquist limit says a sampling rate of $f_s$ represents frequencies up to $f_s/2$, so 16 kHz audio carries information up to 8 kHz, and telephone audio at 8 kHz carries up to 4 kHz. The cell below quantizes to 16 bits and to a deliberately coarse 4 bits so that quantization noise becomes audible in the numbers.

In [3]:
def quantize(sig, n_bits):
    levels = 2 ** n_bits
    step = 2.0 / levels                      # signal is in [-1, 1]
    q = np.round(sig / step) * step
    return np.clip(q, -1.0, 1.0)

for bits in (16, 8, 4):
    q = quantize(x, bits)
    noise = x - q
    snr = 10 * np.log10(np.sum(x ** 2) / np.sum(noise ** 2))
    print(f"{bits:2d}-bit quantization: SNR = {snr:5.1f} dB   (rule of thumb: about {6.02*bits:5.1f} dB)")

print(f"\nNyquist limit at {SR} Hz: {SR/2:.0f} Hz")
print(f"Nyquist limit at 8000 Hz (telephone): 4000 Hz")

16-bit quantization: SNR =  90.4 dB   (rule of thumb: about  96.3 dB)
 8-bit quantization: SNR =  42.5 dB   (rule of thumb: about  48.2 dB)
 4-bit quantization: SNR =  18.6 dB   (rule of thumb: about  24.1 dB)

Nyquist limit at 16000 Hz: 8000 Hz
Nyquist limit at 8000 Hz (telephone): 4000 Hz


## 3. Framing and windowing

Speech is analysed in short frames because its spectrum changes over time. With 25 ms frames and a 10 ms hop at 16 kHz, one frame holds 400 samples and frames overlap by 15 ms. Each frame is multiplied by a window (Hamming or Hann) so that the abrupt cut at the frame edges does not smear energy across the spectrum, an effect called spectral leakage.

In [4]:
frame_len = int(SR * FRAME_MS / 1000)   # 400 samples
hop_len   = int(SR * HOP_MS / 1000)    # 160 samples

def frame_signal(sig, frame_len=frame_len, hop_len=hop_len):
    n_frames = 1 + (len(sig) - frame_len) // hop_len
    idx = np.arange(frame_len)[None, :] + hop_len * np.arange(n_frames)[:, None]
    return sig[idx]

frames = frame_signal(x)
hamming = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(frame_len) / (frame_len - 1))
windowed = frames * hamming
print(f"frame length {frame_len} samples ({FRAME_MS} ms), hop {hop_len} samples ({HOP_MS} ms)")
print(f"{frames.shape[0]} frames x {frames.shape[1]} samples")

frame length 400 samples (25 ms), hop 160 samples (10 ms)
65 frames x 400 samples


In [5]:
# leakage: a rectangular window versus a Hamming window on one frame
f = windowed.shape[0] // 2
rect_spec = 20 * np.log10(np.abs(np.fft.rfft(frames[f], N_FFT)) + 1e-12)
hamm_spec = 20 * np.log10(np.abs(np.fft.rfft(windowed[f], N_FFT)) + 1e-12)
freqs = np.fft.rfftfreq(N_FFT, 1 / SR)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(freqs, rect_spec, lw=0.9, label="rectangular window")
ax.plot(freqs, hamm_spec, lw=0.9, label="Hamming window")
ax.set_xlim(0, 4000); ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("magnitude (dB)")
ax.set_title("Spectral leakage in one 25 ms frame"); ax.legend(); fig.tight_layout()

## 4. The short-time Fourier transform and the spectrogram

The magnitude of the discrete Fourier transform of every frame, stacked over time, is the spectrogram. A 512-point transform at 16 kHz gives 257 one-sided bins spaced 31.25 Hz apart. A long analysis window resolves the individual harmonics of the voice (a narrowband spectrogram); a short window resolves the moments in time and shows formant bands instead (a wideband spectrogram).

In [6]:
def stft_power(sig, win_ms, hop_ms=HOP_MS, n_fft=N_FFT, sr=SR):
    flen = int(sr * win_ms / 1000)
    hlen = int(sr * hop_ms / 1000)
    fr = frame_signal(sig, flen, hlen)
    w = np.hanning(flen)
    spec = np.fft.rfft(fr * w, n_fft, axis=1)
    return (np.abs(spec) ** 2) / n_fft

power = stft_power(x, FRAME_MS)
print("power spectrogram:", power.shape, "= frames x bins")
print("bin spacing:", SR / N_FFT, "Hz;  bins:", power.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, win_ms, name in zip(axes, (40, 6), ("narrowband (40 ms window)", "wideband (6 ms window)")):
    P = stft_power(x, win_ms)
    ax.imshow(10 * np.log10(P.T + 1e-12), origin="lower", aspect="auto",
              extent=[0, len(x) / SR, 0, SR / 2], cmap="magma")
    ax.set_title(name); ax.set_xlabel("time (s)")
axes[0].set_ylabel("frequency (Hz)"); axes[0].set_ylim(0, 5000); fig.tight_layout()

power spectrogram: (65, 257) = frames x bins
bin spacing: 31.25 Hz;  bins: 257


## 5. The mel filterbank and log-mel features

The mel scale spaces filters evenly in a perceptual scale rather than in hertz, so that low frequencies, where speech carries most of its distinguishing information, receive finer resolution. The conversion used here is the common one,

$$m = 2595 \log_{10}\left(1 + \frac{f}{700}\right),\qquad f = 700\left(10^{m/2595} - 1\right).$$

Forty triangular filters are placed with their peaks evenly spaced on the mel scale, each triangle rising from the previous centre to its own and falling to the next. Multiplying the power spectrum by this filterbank and taking the logarithm gives the log-mel features that are the standard input to neural speech models.

In [7]:
def hz_to_mel(f):  return 2595.0 * np.log10(1.0 + f / 700.0)
def mel_to_hz(m):  return 700.0 * (10 ** (m / 2595.0) - 1.0)

def mel_filterbank(n_mels=N_MELS, n_fft=N_FFT, sr=SR, fmin=20.0, fmax=None):
    fmax = fmax or sr / 2
    mels = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels + 2)
    peaks_hz = mel_to_hz(mels)
    bins = np.floor((n_fft + 1) * peaks_hz / sr).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        left, centre, right = bins[m - 1], bins[m], bins[m + 1]
        for k in range(left, centre):
            fb[m - 1, k] = (k - left) / max(centre - left, 1)
        for k in range(centre, right):
            fb[m - 1, k] = (right - k) / max(right - centre, 1)
    return fb, peaks_hz[1:-1]

fb, centres = mel_filterbank()
mel_power = power @ fb.T
log_mel = np.log(mel_power + 1e-10)
print("filterbank:", fb.shape, " log-mel features:", log_mel.shape)
print("first five centre frequencies (Hz):", np.round(centres[:5], 1))
print("last five centre frequencies (Hz): ", np.round(centres[-5:], 1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for row, c in zip(fb, centres):
    axes[0].plot(np.fft.rfftfreq(N_FFT, 1 / SR), row, lw=0.8)
axes[0].set_title("40 triangular mel filters"); axes[0].set_xlabel("frequency (Hz)")
axes[1].imshow(log_mel.T, origin="lower", aspect="auto", cmap="magma",
               extent=[0, len(x) / SR, 0, N_MELS])
axes[1].set_title("log-mel features"); axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("mel filter")
fig.tight_layout()

filterbank: (40, 257)  log-mel features: (65, 40)
first five centre frequencies (Hz): [ 65.1 113.1 164.  218.1 275.7]
last five centre frequencies (Hz):  [5720.2 6122.4 6549.9 7004.2 7487. ]


## 6. Cepstral coefficients and deltas

The log-mel values of neighbouring filters are strongly correlated. A discrete cosine transform decorrelates them and concentrates the shape of the spectral envelope in the first few coefficients; keeping about thirteen of them gives the classic MFCC front end. Adding first and second differences (deltas and delta-deltas) brings the count to 39, the size quoted in the chapter. The type-II DCT is written out here rather than imported, so the exact computation is visible.

In [8]:
def dct_ii(mat, n_out=N_MFCC):
    n = mat.shape[1]
    k = np.arange(n_out)[:, None]
    n_idx = np.arange(n)[None, :]
    basis = np.cos(np.pi * k * (2 * n_idx + 1) / (2 * n))
    basis *= np.sqrt(2.0 / n)
    basis[0] *= 1 / np.sqrt(2)
    return mat @ basis.T

def deltas(feat, width=2):
    pad = np.pad(feat, ((width, width), (0, 0)), mode="edge")
    denom = 2 * sum(i * i for i in range(1, width + 1))
    out = np.zeros_like(feat)
    for i in range(1, width + 1):
        out += i * (pad[width + i: width + i + len(feat)] - pad[width - i: width - i + len(feat)])
    return out / denom

mfcc = dct_ii(log_mel)
d1, d2 = deltas(mfcc), deltas(deltas(mfcc))
features = np.hstack([mfcc, d1, d2])
print("MFCC:", mfcc.shape, " with deltas:", features.shape)

# the sizes quoted in Chapter 3 for one 25 ms frame at 16 kHz
print(f"\n400 samples -> {N_FFT}-point transform -> {N_FFT//2+1} power bins -> {N_MELS} mel -> {N_MELS} log -> {N_MFCC} MFCC ({3*N_MFCC} with deltas)")

# how much of the log-mel variance the first coefficients carry
var = mfcc.var(axis=0)
print("share of cepstral variance in coefficients 1-5:", round(float(var[1:6].sum() / var[1:].sum()), 3))

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.imshow(mfcc.T, origin="lower", aspect="auto", cmap="viridis", extent=[0, len(x)/SR, 0, N_MFCC])
ax.set_title("MFCCs"); ax.set_xlabel("time (s)"); ax.set_ylabel("coefficient"); fig.tight_layout()

MFCC: (65, 13)  with deltas: (65, 39)

400 samples -> 512-point transform -> 257 power bins -> 40 mel -> 40 log -> 13 MFCC (39 with deltas)
share of cepstral variance in coefficients 1-5: 0.989


**Cepstral mean and variance normalization (CMVN).** Chapter 3 stresses that the scope of normalization (per utterance, per speaker, global) is a decision to report, not a fixed rule. Here it is applied per utterance.

In [9]:
cmvn = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-8)
print("after CMVN: mean ~", np.round(cmvn.mean(), 6), " std ~", np.round(cmvn.std(), 6))

after CMVN: mean ~ -0.0  std ~ 1.0


## 7. Linear prediction

Linear predictive coding models each sample as a weighted sum of the previous $p$ samples. The coefficients follow from the autocorrelation of a windowed frame through the Levinson-Durbin recursion, and the roots of the resulting polynomial give the resonances of the vocal tract. With the synthetic vowel the answer is known in advance: the formants were set to 700, 1220 and 2600 Hz.

In [10]:
def autocorr(frame, order):
    r = np.correlate(frame, frame, mode="full")[len(frame) - 1:]
    return r[: order + 1]

def levinson_durbin(r, order):
    a = np.zeros(order + 1); a[0] = 1.0
    e = r[0]
    for i in range(1, order + 1):
        acc = r[i] + np.dot(a[1:i], r[i - 1:0:-1]) if i > 1 else r[1]
        k = -acc / e
        a_new = a.copy()
        for j in range(1, i):
            a_new[j] = a[j] + k * a[i - j]
        a_new[i] = k
        a = a_new
        e *= (1 - k * k)
    return a, e

order = 12
frame = windowed[len(windowed) // 2] * 1.0
frame = lfilter([1.0, -0.97], [1.0], frame)          # pre-emphasis
a, err = levinson_durbin(autocorr(frame, order), order)
roots = np.roots(a)
roots = roots[np.imag(roots) > 0]
freqs_hz = np.sort(np.angle(roots) * SR / (2 * np.pi))
bw_hz = -0.5 * (SR / (2 * np.pi)) * np.log(np.abs(roots[np.argsort(np.angle(roots))]))
keep = (freqs_hz > 90) & (bw_hz < 400)
print("LPC order:", order, " residual energy:", round(float(err), 6))
print("estimated formants (Hz):", np.round(freqs_hz[keep][:3], 1), " (synthesised at 700, 1220, 2600)")

w, h = np.linspace(0, SR / 2, 512), None
lpc_spec = 20 * np.log10(np.abs(1.0 / np.fft.rfft(a, 1024))[:512] + 1e-12)
fft_spec = 20 * np.log10(np.abs(np.fft.rfft(frame, 1024))[:512] + 1e-12)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(w, fft_spec - fft_spec.max(), lw=0.8, label="frame spectrum")
ax.plot(w, lpc_spec - lpc_spec.max(), lw=1.6, label=f"LPC envelope (order {order})")
ax.set_xlim(0, 5000); ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("relative magnitude (dB)")
ax.legend(); ax.set_title("Linear prediction as a spectral envelope"); fig.tight_layout()

LPC order: 12  residual energy: 0.014938
estimated formants (Hz): [ 755.6 1241.  2583.2]  (synthesised at 700, 1220, 2600)


## 8. Telephone band and resampling

Chapter 3 shows one recording in two forms: the original 16 kHz audio and a simulated telephone version, band-limited to 300 to 3400 Hz and resampled to 8 kHz. Resampling the telephone version back to 16 kHz does not restore what the filter removed, which is why the upper mel bands of telephone audio carry almost nothing and why a recogniser for call-centre audio should be trained or adapted on matched data.

In [11]:
sos = butter(8, [300 / (SR / 2), 3400 / (SR / 2)], btype="band", output="sos")
tel = sosfilt(sos, x)
tel_8k = resample_poly(tel, 8000, SR)
tel_back = resample_poly(tel_8k, SR, 8000)[: len(x)]

def band_energy(sig, lo, hi, sr=SR):
    P = stft_power(sig, FRAME_MS, sr=sr)
    f = np.fft.rfftfreq(N_FFT, 1 / sr)
    sel = (f >= lo) & (f < hi)
    return float(P[:, sel].sum())

for name, sig in (("original 16 kHz", x), ("telephone, back at 16 kHz", tel_back)):
    below = band_energy(sig, 300, 3400)
    above = band_energy(sig, 4000, 8000)
    print(f"{name:26s}  energy 300-3400 Hz: {below:10.2f}   energy 4-8 kHz: {above:10.2f}")

mel_tel = np.log(stft_power(tel_back, FRAME_MS) @ fb.T + 1e-10)
print("\nmean log-mel value, top five filters:")
print("  original :", np.round(log_mel[:, -5:].mean(axis=0), 2))
print("  telephone:", np.round(mel_tel[:, -5:].mean(axis=0), 2))

original 16 kHz             energy 300-3400 Hz:     401.61   energy 4-8 kHz:       1.18
telephone, back at 16 kHz   energy 300-3400 Hz:     402.17   energy 4-8 kHz:       0.00

mean log-mel value, top five filters:
  original : [-11.78 -11.66 -11.69 -11.66 -11.56]
  telephone: [-20.4  -20.16 -16.11 -15.08 -15.41]


## 9. Optional: a clip from Common Voice Arabic

Chapter 3 suggests validated Common Voice Arabic clips for front-end experiments, and asks that the release version and validated-hour count be reported whenever they are used. The cell below needs internet access and the `datasets` package, so it is left commented out; everything above runs without it.

In [12]:
# !pip install datasets soundfile
# from datasets import load_dataset
# ds = load_dataset("mozilla-foundation/common_voice_17_0", "ar", split="validated", streaming=True)
# clip = next(iter(ds))
# audio = clip["audio"]["array"]; sr_in = clip["audio"]["sampling_rate"]
# x = resample_poly(audio, SR, sr_in) if sr_in != SR else audio
# x = (x / np.max(np.abs(x))).astype(np.float32)
# print(clip["sentence"], "|", len(x) / SR, "s")
# # then re-run sections 3 to 7 on this clip
# # report: release version, split, and validated hours used

## What to report

Chapter 3 asks that a feature pipeline be documented well enough for another researcher to reproduce it. For the settings used above that means: sampling rate 16 kHz; pre-emphasis 0.97 (linear prediction section only); 25 ms frames with a 10 ms hop; Hamming or Hann window; 512-point transform giving 257 bins; 40 mel filters from 20 Hz to 8 kHz; natural-logarithm compression; 13 cepstral coefficients with deltas and delta-deltas; and CMVN applied per utterance.